In [ ]:
Threads.nthreads()

In [ ]:
using Pkg
Pkg.activate("..")
using StaticArrays
using Eliashberg
using GLMakie
GLMakie.activate!()
println("Eliashberg loaded successfully!")

### Define a 1D Tight-Binding model (Perfect nesting condition)


In [ ]:
lattice = ChainLattice(1.5) # 晶格常数 1.5 的一维链
kgrid = generate_reciprocal_lattice(lattice, 100)

tb_model = TightBinding(lattice, 1.0, -0.3)
tb = tb_model


In [ ]:
dispersion_data = compute_dispersion_curve_data(tb, kgrid)
f = plot_dispersion_curves(dispersion_data.coordinates, dispersion_data.bands)
f


In [ ]:
Q = SVector{1, Float64}(pi)
cdw = ChargeDensityWave(Q)
interaction = ConstantInteraction(-2.5)

phi_values = range(0.0, 1.5, length=30)
T_val = 0.1

F_exact_vals = evaluate_action(phi_values, cdw, tb_model, interaction, kgrid, ExactTrLn(); T=T_val)
F_RPA_vals = evaluate_action(phi_values, cdw, tb_model, interaction, kgrid, RPA(); T=T_val)
phi_gs_exact = solve_ground_state(cdw, tb_model, interaction, kgrid, ExactTrLn(); phi_guess=0.5, T=T_val)
phi_gs_rpa = solve_ground_state(cdw, tb_model, interaction, kgrid, RPA(); phi_guess=0.5, T=T_val)

println("Exact Ground state order parameter ϕ = ", phi_gs_exact)
println("RPA Ground state order parameter ϕ = ", phi_gs_rpa)


In [ ]:
gapped_tb = MeanFieldDispersion(tb_model, cdw, phi_gs_exact)


In [ ]:
dispersion_data = compute_dispersion_curve_data(gapped_tb, kgrid)
f = plot_dispersion_curves(dispersion_data.coordinates, dispersion_data.bands)
save("renormalized_1d_eband.png", f)
f


In [ ]:

fig_action = Figure()
ax = Axis(fig_action[1, 1], xlabel = "ϕ", ylabel = "Free Energy F", title="CDW Action: Exact vs RPA")
lines!(ax, phi_values, F_exact_vals, label="Exact Tr[ln]", linewidth=2)
lines!(ax, phi_values, F_RPA_vals, label="RPA", linestyle=:dash, linewidth=2)
hlines!(ax, [0.0], color=:black)
axislegend(ax)
save("tb_1d_effective_pot.png", fig_action)
fig_action

In [ ]:
Nq = 400
qgrid = generate_1d_kgrid(Nq)
landscape_data = scan_instability_landscape(tb_model, kgrid, qgrid; T=T_val)
landscape_axes = compute_landscape_line_data(qgrid, landscape_data)
fig_ls = plot_landscape(Val(1), landscape_axes.qs, landscape_axes.values)
save("tb_1d_rpa_static.png", fig_ls)
display(fig_ls)


In [ ]:
# 1. 定义高对称路径 (例如：Γ -> X -> M -> Γ)
nodes = [
    SVector{1, Float64}(0.0),    # Γ
    SVector{1, Float64}(pi),     # X
]
labels = ["Γ", "X"]
qpath = generate_kpath(nodes, labels; n_pts_per_segment=500)
# 2. 定义能量/频率范围 ω
omegas = range(0.0, 5, length=1000)
# 3. 执行动态谱扫描
# A(q, ω) 矩阵的大小为 (length(q_path), length(omegas))
spectral_matrix = scan_spectral_function(tb_model, kgrid, qpath, omegas; T=T_val, η=0.04)
# 4. 可视化扫描结果
# 该函数会自动根据 q_path 的节点添加高对称点标签和垂直分割线
fig_spectral = plot_spectral_function(qpath, omegas, spectral_matrix)
save("tb_1d_rpa_spectral.png", fig_spectral)
display(fig_spectral)


### Define a 2D Tight-Binding model (Perfect nesting condition)


In [ ]:
t = 1.0
tp = 0.0
mu = 0.0
lattice = SquareLattice(1.0)
tb_model = TightBinding(lattice, t, tp, mu)
kgrid = generate_2d_kgrid(100, 100)
path = generate_kpath(lattice; n_pts_per_segment=50)

Nq = 100
qgrid = generate_2d_kgrid(Nq, Nq)


In [ ]:
CairoMakie.activate!()
band_data = compute_path_band_data(tb_model, path)
f = plot_band_structure(path, band_data.bands)
save("tb_2d_eband_GXMG.png", f)
f


In [ ]:
dispersion_data = compute_dispersion_surface_data(tb_model, kgrid)
f = plot_dispersion_surface(dispersion_data.kxs, dispersion_data.kys, dispersion_data.energy_matrix)
save("tb_2d_eband.png", f)
f


In [ ]:
Q = SVector{2, Float64}(pi, pi)
cdw = ChargeDensityWave(Q)
interaction = ConstantInteraction(2.0)

phi_values = range(0.0, 1.5, length=30)
T_val = 0.1

F_exact_vals = evaluate_action(phi_values, cdw, tb_model, interaction, kgrid, ExactTrLn(); T=T_val)
F_RPA_vals = evaluate_action(phi_values, cdw, tb_model, interaction, kgrid, RPA(); T=T_val)
phi_gs_exact = solve_ground_state(cdw, tb_model, interaction, kgrid, ExactTrLn(); phi_guess=0.5, T=T_val)
phi_gs_rpa = solve_ground_state(cdw, tb_model, interaction, kgrid, RPA(); phi_guess=0.5, T=T_val)

println("Exact Ground state order parameter ϕ = ", phi_gs_exact)
println("RPA Ground state order parameter ϕ = ", phi_gs_rpa)


In [ ]:
fig_action = Figure()
ax = Axis(fig_action[1, 1], xlabel = "ϕ", ylabel = "Free Energy F", title="CDW Action: Exact vs RPA")
lines!(ax, phi_values, F_exact_vals, label="Exact Tr[ln]", linewidth=2)
lines!(ax, phi_values, F_RPA_vals, label="RPA", linestyle=:dash, linewidth=2)
hlines!(ax, [0.0], color=:black)
axislegend(ax)
save("tb_2d_effective_pot.png", fig_action)

fig_action

In [ ]:
landscape_matrix = scan_instability_landscape(tb_model, kgrid, qgrid; T=T_val)
landscape_axes = compute_landscape_axes(qgrid)
fig_ls = plot_landscape(Val(2), landscape_axes.qxs, landscape_axes.qys, landscape_matrix)


In [ ]:
# 1. 定义高对称路径 (例如：Γ -> X -> M -> Γ)
# 定义 3D 简立方的高对称点
nodes = [
    SVector{2, Float64}(0, 0),    # Γ
    SVector{2, Float64}(pi, 0),   # X
    # SVector{2, Float64}(pi, pi),  # M
    # SVector{2, Float64}(0, 0)     # Γ
]
labels = ["Γ", "X"]
qpath = generate_kpath(nodes, labels; n_pts_per_segment=5000)

# 2. 定义能量/频率范围 ω
omegas = range(0.0, 5, length=2000)
# 3. 执行动态谱扫描
# A(q, ω) 矩阵的大小为 (length(q_path), length(omegas))
spectral_matrix = scan_spectral_function(tb_model, kgrid, qpath, omegas; T=T_val, η=0.04)
# 4. 可视化扫描结果
# 该函数会自动根据 q_path 的节点添加高对称点标签和垂直分割线
fig_spectral = plot_spectral_function(qpath, omegas, spectral_matrix)
display(fig_spectral)

### Define a 3D Tight-Binding model (Perfect nesting condition)


In [ ]:
t = 1.0
mu = 0.0
lattice = CubicLattice(1.0)
tb_model = TightBinding(lattice, t, mu)
kgrid = generate_3d_kgrid(50, 50, 50)
path = generate_kpath(lattice; n_pts_per_segment=50)


In [ ]:
CairoMakie.activate!()
band_data = compute_path_band_data(tb_model, path)
f = plot_band_structure(path, band_data.bands)
save("tb_3d_eband_GXMGR.png", f)


In [ ]:
GLMakie.activate!()
fermi_surface = compute_fermi_surface_volume(tb_model; n_pts=60)
f = plot_fermi_surface(fermi_surface.kxs, fermi_surface.kys, fermi_surface.kzs, fermi_surface.energy_volume; E_Fermi=0.0)
# save("tb_3d_eband_fs.png", f)
